# PointDiT in motion: Kaggle runner

Needs a GPU accelerator, Internet on, the "PointDiT TUM RGB-D Subset" dataset attached, a Kaggle secret `GITHUB_TOKEN` while the repo is private (fine-grained, read-only Contents on this repo), and a Kaggle secret `HF_TOKEN` for an account with access to `facebook/dinov3-vitl16-pretrain-lvd1689m`.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/Pointdit-In-Motion.git"
DINOV3_COMMIT = "6876159a11b4df116f30f667f8c9888617df0751"
OUTPUT_DIR = "/kaggle/working/outputs"
PROJECT_BRANCH = "feat/temporal-consistency"

PROJECT_DIR = "/kaggle/working/Pointdit-In-Motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
DINOV3_DIR = f"{FORK_DIR}/third_party/dinov3"

In [ ]:
import base64
import os

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Only needed while the repo is private. Git reads the token from the environment, so it
# never appears in a command, in cell output, or in the clone's .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

# Check for .git, not just the directory: after a failed clone, the DINOv3 clone below
# creates PROJECT_DIR as a parent, which would otherwise skip this clone on every rerun.
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules --branch {PROJECT_BRANCH} {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"

In [ ]:
if not os.path.isdir(DINOV3_DIR):
    !git clone -q https://github.com/facebookresearch/dinov3.git {DINOV3_DIR}
if DINOV3_COMMIT:
    !git -C {DINOV3_DIR} checkout -q {DINOV3_COMMIT}
else:
    print("DINOV3_COMMIT is empty; paste the hash below into the config cell to pin it.")
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"

In [ ]:
!pip install -q -r {FORK_DIR}/requirements.txt -r {PROJECT_DIR}/requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python {PROJECT_DIR}/scripts/setup_weights.py

In [ ]:
import sys

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences

sequences = sorted(find_sequences("/kaggle/input"), key=lambda s: s.name)
for sequence in sequences:
    print(sequence)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/smoke_test.py --sequence "{sequences[0]}" --output-dir {OUTPUT_DIR}/smoke

In [ ]:
import torch

print("torch  ", torch.__version__)
print("CUDA   ", torch.version.cuda)
print("GPU    ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"
!echo "dinov3  $(git -C {DINOV3_DIR} rev-parse HEAD)"

In [ ]:
REFERENCE_SEQUENCE = "rgbd_dataset_freiburg1_360"
REFERENCE_DIR = f"{PROJECT_DIR}/tests/fixtures/smoke_fr1360_664736a"

from pim.data.tum import read_rgb_list

ref_seq = next((s for s in sequences if s.name == REFERENCE_SEQUENCE), None)
if ref_seq is None:
    raise FileNotFoundError(f"{REFERENCE_SEQUENCE} not found under /kaggle/input")
first_frame = read_rgb_list(ref_seq)[0].path
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/check_sampler_init.py --image "{first_frame}" --reference "{REFERENCE_DIR}"

In [ ]:
EVAL_DATA_DIR = "/kaggle/working/datasets/eval"
BOOSTER_DIR = f"{EVAL_DATA_DIR}/Booster"

if not os.path.isdir(BOOSTER_DIR):
    from huggingface_hub import hf_hub_download

    hf_hub_download("haofeixu/pointdit-eval", "Booster.zip", repo_type="dataset", local_dir=EVAL_DATA_DIR)
    !cd {EVAL_DATA_DIR} && unzip -qo Booster.zip
!CUDA_VISIBLE_DEVICES=0 python {PROJECT_DIR}/scripts/check_booster_metrics.py --booster-dir {BOOSTER_DIR}

In [ ]:
import json
import os
import subprocess
import sys
import threading
from pathlib import Path

import yaml

from pim.data.tum import apply_window, load_sequence

RUN_CONFIG = f"{PROJECT_DIR}/configs/baseline.yaml"
SEQUENCES_CONFIG = f"{PROJECT_DIR}/configs/sequences.yaml"
PREDICTIONS_DIR = "/kaggle/working/predictions"

data_roots = {s.parent for s in sequences}
if len(data_roots) != 1:
    raise RuntimeError(f"sequences are spread over several folders: {sorted(data_roots)}")
DATA_ROOT = data_roots.pop()
names = list(yaml.safe_load(open(SEQUENCES_CONFIG))["sequences"])
frame_counts = {n: len(apply_window(load_sequence(DATA_ROOT / n, SEQUENCES_CONFIG))) for n in names}
if len(names) != 4:
    raise RuntimeError(f"expected four sequences to split over two GPUs, got {names}")
# Largest with smallest is the most even two-and-two split.
by_size = sorted(names, key=frame_counts.get, reverse=True)
assignment = {0: [by_size[0], by_size[3]], 1: [by_size[1], by_size[2]]}


def launch(gpu: int, seq_names: list[str]) -> tuple[subprocess.Popen, threading.Thread]:
    cmd = [sys.executable, f"{PROJECT_DIR}/scripts/predict.py", "--config", RUN_CONFIG,
           "--sequences-config", SEQUENCES_CONFIG, "--data-root", str(DATA_ROOT),
           "--sequences", *seq_names, "--out", PREDICTIONS_DIR, "--device", "cuda:0"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, env=env, cwd=PROJECT_DIR, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    def relay() -> None:
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="", flush=True)

    thread = threading.Thread(target=relay, daemon=True)
    thread.start()
    return proc, thread


for gpu, seq_names in assignment.items():
    print(f"gpu{gpu}: {seq_names} ({sum(frame_counts[n] for n in seq_names)} frames)")
workers = {gpu: launch(gpu, seq_names) for gpu, seq_names in assignment.items()}
for proc, thread in workers.values():
    proc.wait()
    thread.join()
failed = {gpu: proc.returncode for gpu, (proc, _) in workers.items() if proc.returncode != 0}
if failed:
    raise RuntimeError(f"predict.py failed on {failed} (gpu: exit code); see the output above")

run_dir = Path(PREDICTIONS_DIR) / yaml.safe_load(open(RUN_CONFIG))["name"]
run = json.loads((run_dir / "run.json").read_text())
print(f"\n{run_dir}: {run['start_time']} -> {run['end_time']}, "
      f"{run['frames_predicted']} predicted, {run['frames_skipped']} skipped")
for name in names:
    seq = run["sequences"].get(name)
    files = len(list((run_dir / name).glob("*.npz")))
    if seq is None or files != seq["frames"] or seq["frames"] != frame_counts[name]:
        raise RuntimeError(f"{name}: run.json entry {seq}, {files} npz files, {frame_counts[name]} frames in window")
    print(f"  {name}: window {seq['window']}, {files} files, "
          f"mean latency {seq['mean_latency_ms']:.0f} ms, peak {seq['peak_mem_gb']:.2f} GB")
for inv in run["invocations"]:
    commits = ", ".join(f"{k} {v['commit'][:7]}{'+dirty' if v['dirty'] else ''}" for k, v in inv["commits"].items())
    print(f"  run {inv['id']}: GPU {inv['cuda_visible_devices']} {inv['gpu']}, torch {inv['torch']}, "
          f"CUDA {inv['cuda']}, {inv['start_time']} -> {inv['end_time']}, {commits}")

In [ ]:
PREVIEW_FRAMES = {
    "rgbd_dataset_freiburg1_desk": [209, 359, 508],
    "rgbd_dataset_freiburg1_room": [366, 516, 665],
    "rgbd_dataset_freiburg1_360": [0, 150, 299],
    "rgbd_dataset_freiburg3_long_office_household": [637, 787, 936],
}
for seq, frames in PREVIEW_FRAMES.items():
    frames_arg = " ".join(map(str, frames))
    !python {PROJECT_DIR}/scripts/preview_predictions.py --predictions /kaggle/working/predictions/baseline --data-root "{DATA_ROOT}" --sequences-config {PROJECT_DIR}/configs/sequences.yaml --sequence {seq} --frames {frames_arg} --out /kaggle/working/outputs/preview